## Camera-Level Detectability Table

This notebook creates a detectability table for each camera location by comparing:

- **Predicted species** from iucn_clipped_cleaned_ssusa_consistent_500g.geojson
- **Observed species** from SSUSA observations
- **Camera footprints** from ssusa_camera_footprints_1km.geojson

For each camera (coord_id), we calculate:

- number of predicted species
- number of observed species
- number of matched species
- detectability = matched / predicted

In [4]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd

In [5]:
BUFFER_KM = 1

BASE_SSUSA_PATH = "../../data/ssusa"
OUTPUT_PATH = "../../outputs"

# SSUSA observations CSV
SSUSA_CSV = "cleaned_snapshot_usa_iucn.csv"

# Cleaned IUCN file already made SSUSA-consistent and filtered to >=500 g
IUCN_DATA = "iucn_clipped_cleaned_ssusa_consistent_500g.geojson"

# Camera footprints
CAMERA_FOOTPRINTS = f"ssusa_camera_footprints_{BUFFER_KM}km.geojson"

In [7]:
ssusa = pd.read_csv(os.path.join(BASE_SSUSA_PATH, SSUSA_CSV))

iucn = gpd.read_file(os.path.join(OUTPUT_PATH, IUCN_DATA))
camera_fp = gpd.read_file(os.path.join(OUTPUT_PATH, CAMERA_FOOTPRINTS))

print("SSUSA shape:", ssusa.shape)
print("IUCN shape:", iucn.shape)
print("Camera footprints shape:", camera_fp.shape)

/var/folders/dx/fz7lq92d3zz2d72jhk9_rryr0000gn/T/ipykernel_58761/3479300449.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  ssusa = pd.read_csv(os.path.join(BASE_SSUSA_PATH, SSUSA_CSV))


SSUSA shape: (698887, 26)
IUCN shape: (130, 29)
Camera footprints shape: (7303, 3)


In [8]:
# Step 1: Function to create a coordinate-based site ID
# This avoids precision mismatch between datasets
# ------------------------------------------------------------
def make_coord_id(df, lon_col="Longitude", lat_col="Latitude", decimals=8):
    return (
        df[lon_col].round(decimals).astype(str) + "_"
        + df[lat_col].round(decimals).astype(str)
    )

In [9]:
# Step 2: Create coord_id in camera footprints
# Each row in camera_fp is one camera site
# ------------------------------------------------------------
camera_fp["coord_id"] = make_coord_id(camera_fp)
ssusa["coord_id"] = make_coord_id(ssusa)

print("Unique camera footprints:", camera_fp["coord_id"].nunique())
print("Unique SSUSA camera IDs:", ssusa["coord_id"].nunique())

Unique camera footprints: 7303
Unique SSUSA camera IDs: 7303


In [10]:
# Attach footprint geometry to SSUSA observations
# ------------------------------------------------------------
ssusa_fp = ssusa.merge(
    camera_fp[["coord_id", "geometry"]],
    on="coord_id",
    how="left"
)

print("Rows missing footprint geometry:", ssusa_fp["geometry"].isna().sum())

Rows missing footprint geometry: 0


### Spatial Join Between SSUSA Footprint and IUCN 

In [11]:
# Step 1: Create GeoDataFrame from merged SSUSA + camera footprints
# We can use the geometry from camera_fp for each SSUSA observation
ssusa_fp = gpd.GeoDataFrame(
    ssusa_fp,
    geometry="geometry",
    crs=camera_fp.crs
)

if ssusa_fp.crs != iucn.crs:
    ssusa_fp = ssusa_fp.to_crs(iucn.crs)

print("SSUSA CRS:", ssusa_fp.crs)
print("IUCN CRS:", iucn.crs)


SSUSA CRS: EPSG:5070
IUCN CRS: EPSG:5070


In [103]:
# Step 2: Ensure SSUSA footprints and IUCN are in the same CRS
# Spatial join requires matching CRS
# ------------------------------------------------------------
print("ssusa_fp CRS:", ssusa_fp.crs)
print("iucn CRS:", iucn.crs)

if ssusa_fp.crs != iucn.crs:
    ssusa_fp = ssusa_fp.to_crs(iucn.crs)

print("ssusa_fp CRS after check:", ssusa_fp.crs)
print("iucn CRS:", iucn.crs)

ssusa_fp CRS: EPSG:5070
iucn CRS: EPSG:5070
ssusa_fp CRS after check: EPSG:5070
iucn CRS: EPSG:5070


In [12]:
## Build predicted species list for each camera
camera_pred = gpd.sjoin(
    camera_fp[["coord_id", "geometry"]].drop_duplicates(),
    iucn[["sci_name", "geometry"]],
    how="inner",
    predicate="intersects"
)[["coord_id", "sci_name"]].drop_duplicates()

camera_pred.head()

,coord_id,sci_name
0,-136.2225_59.42643,canis latrans
0,-136.2225_59.42643,vulpes vulpes
0,-136.2225_59.42643,ursus americanus
0,-136.2225_59.42643,erethizon dorsatum
0,-136.2225_59.42643,lepus americanus


In [ ]:
# print count of species in IUCN and SSUSA_fp 
print("Camera × predicted-species rows:", len(camera_pred))
print("Cameras with >=1 predicted species:", camera_pred["coord_id"].nunique())
print("Unique predicted species:", camera_pred["sci_name"].nunique())

Unique species in IUCN: 115
Unique species in SSUSA_fp: 113


In [13]:
camera_obs = (
    ssusa_fp[["coord_id", "Sci_Name"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"Sci_Name": "sci_name"})
)

camera_obs.head()

,coord_id,sci_name
0,-136.2225_59.42643,ursus arctos
55,-136.2225_59.42643,alces alces
99,-135.9288_59.39905,canis latrans
101,-135.9288_59.39905,ursus arctos
103,-135.9288_59.39905,lynx canadensis


In [106]:
# Step 3: Spatial join SSUSA footprints with IUCN polygons
# This assigns IUCN-predicted species to each camera site
# ------------------------------------------------------------

ssusa_iucn = gpd.sjoin(
    ssusa_fp[["coord_id", "geometry"]].drop_duplicates(),
    iucn[["sci_name", "geometry"]],
    how="inner",
    predicate="intersects"
)

ssusa_iucn.head()

,coord_id,geometry,index_right,sci_name
0,-136.2225_59.42643,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",75,canis latrans
0,-136.2225_59.42643,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",124,vulpes vulpes
0,-136.2225_59.42643,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",100,ursus americanus
0,-136.2225_59.42643,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",105,erethizon dorsatum
0,-136.2225_59.42643,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",119,lepus americanus


### Count Predicted Sites per species 

In [107]:
iucn_pred = (
    ssusa_iucn[["coord_id", "sci_name"]]
    .drop_duplicates()
    .groupby("sci_name")
    .size()
    .reset_index(name="predicted_sites")
)

iucn_pred.head()

,sci_name,predicted_sites
0,alces alces,834
1,alexandromys oeconomus,42
2,antilocapra americana,630
3,aplodontia rufa,251
4,bassariscus astutus,1208


### Count Observed sites per species (SSUSA)

In [108]:
# Count number of camera sites where SSUSA observed each species
# Each species counted once per camera site
# ------------------------------------------------------------

ssusa_obs = (
    ssusa_fp[["coord_id", "Sci_Name"]]
    .drop_duplicates()        # ensures one species counted once per site
    .groupby("Sci_Name")
    .size()
    .reset_index(name="observed_sites")
)

ssusa_obs.head()

,Sci_Name,observed_sites
0,alces alces,165
1,ammospermophilus harrisii,17
2,ammospermophilus leucurus,56
3,antilocapra americana,130
4,aplodontia rufa,13


### Create Detectability Table

In [109]:
iucn_pred = iucn_pred.rename(columns={"sci_name": "species"})
ssusa_obs = ssusa_obs.rename(columns={"Sci_Name": "species"})

detect_table = pd.merge(
    iucn_pred,
    ssusa_obs,
    on="species",
    how="left"
)

detect_table["observed_sites"] = detect_table["observed_sites"].fillna(0)
detect_table["observed_sites"] = detect_table["observed_sites"].astype(int)

In [110]:
detect_table["detectability"] = (
    detect_table["observed_sites"] /
    detect_table["predicted_sites"]
).round(6)

In [111]:
print("Number of IUCN species:", detect_table["species"].nunique())

Number of IUCN species: 115


In [112]:
detect_table.head(20)

,species,predicted_sites,observed_sites,detectability
0,alces alces,834,165,0.197842
1,alexandromys oeconomus,42,0,0.000000
2,antilocapra americana,630,130,0.206349
3,aplodontia rufa,251,13,0.051793
4,bassariscus astutus,1208,21,0.017384
5,callospermophilus lateralis,615,6,0.009756
6,callospermophilus saturatus,32,0,0.000000
7,canis latrans,7099,3136,0.441752
8,canis rufus,34,5,0.147059
9,cervus canadensis,858,325,0.378788


In [113]:
# ------------------------------------------------------------
# Save species detectability table as CSV
# ------------------------------------------------------------

output_file = os.path.join(
    OUTPUT_PATH,
    "species_detectability_table.csv"
)

detect_table.to_csv(output_file, index=False)

print("Saved to:", output_file)

Saved to: ../../outputs/species_detectability_table.csv
